# Targeted amplification analysis

Run `python scripts/run.py targeted-amplification` first. Validation sweeps show the multiplier search; test tables report the multiplier selected on validation for each feature group. Do not choose feature groups or models by test scores.


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "src" / "sae_emotion" / "__init__.py").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the sae-emotion repository.")
sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)


In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_DIR = Path("data")
OUTPUT_DIR = DATA_DIR / "targeted_amplification_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def load_results(pattern):
    frames = []
    for path in sorted(DATA_DIR.glob(pattern)):
        if path.name.startswith("multiplier_sweep") and not re.fullmatch(
            r"multiplier_sweep_run\d+_[^_]+_L\d+_n\d+\.csv", path.name
        ):
            continue
        frame = pd.read_csv(path)
        frame["model_setting"] = path.parent.parent.name.removesuffix("-feature-act")
        frame["run_file"] = path.name
        frame["source_file"] = str(path)
        frames.append(frame)
    if not frames:
        raise FileNotFoundError(f"No results match {pattern}. Run targeted-amplification first.")
    return pd.concat(frames, ignore_index=True)


validation = load_results("*feature-act*/auto_weak_emotion_boosting/multiplier_sweep_run*.csv")
test = load_results("*feature-act*/auto_weak_emotion_boosting/selected_test_run*_metrics.csv")
validation.to_csv(OUTPUT_DIR / "validation_sweeps.csv", index=False)
test.to_csv(OUTPUT_DIR / "selected_test_results.csv", index=False)
display(test[["model_setting", "run_file", "multiplier", "validation_macro_f1",
              "before_macro_f1", "after_macro_f1", "delta_macro_f1"]])


In [ ]:
# Validation curves are tuning diagnostics, not final test results.
fig, ax = plt.subplots(figsize=(10, 5))
for (model, run), group in validation.groupby(["model_setting", "run_file"]):
    group = group.sort_values("multiplier")
    ax.plot(group["multiplier"], group["after_macro_f1"], marker="o", label=f"{model}: {run}")
ax.set(xlabel="Multiplier", ylabel="Validation macro-F1")
ax.legend(fontsize=6, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.savefig(OUTPUT_DIR / "validation_multiplier_sweeps.png", bbox_inches="tight", dpi=200)
plt.show()

per_emotion = load_results("*feature-act*/auto_weak_emotion_boosting/selected_test_run*_per_emotion.csv")
per_emotion.to_csv(OUTPUT_DIR / "selected_test_per_emotion.csv", index=False)
display(per_emotion)
